In [2]:
# Import necessary libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import glob

# Function to open a zarr file
def open_zarr_file(zarr_path):
    """Open a zarr file and return the dataset"""
    try:
        ds = xr.open_zarr(zarr_path)
        print(f"Dataset: {os.path.basename(zarr_path)}")
        print(f"Dimensions: {dict(ds.dims)}")
        print("Variables:", list(ds.data_vars.keys()))
        return ds
    except Exception as e:
        print(f"Error opening {zarr_path}: {str(e)}")
        return None

In [3]:
# Function to find all zarr files in a directory
def find_zarr_files(directory_path):
    """Find all .zarr files in a directory"""
    zarr_pattern = os.path.join(directory_path, "*.zarr")
    zarr_files = glob.glob(zarr_pattern)
    zarr_files.sort()  # Sort files for consistent ordering
    return zarr_files

# Function to enhance image contrast
def enhance_contrast(data, enhancement_factor=3.0):
    """Apply contrast enhancement to an image band"""
    valid_data = data.values[~np.isnan(data.values)]
    if len(valid_data) == 0:
        return np.zeros_like(data)
    
    p_low = max(0, 50 - enhancement_factor * 25)
    p_high = min(100, 50 + enhancement_factor * 25)
    
    vmin = np.nanpercentile(valid_data, p_low)
    vmax = np.nanpercentile(valid_data, p_high)
    
    if vmin == vmax:
        vmin = vmin - 0.1
        vmax = vmax + 0.1
    
    normalized = (data - vmin) / (vmax - vmin)
    return np.clip(normalized, 0, 1)



In [4]:
# Function to create RGB composites for all timesteps
def visualize_all_timesteps(ds, var_name='Rad', rgb_bands=[0, 1, 4], 
                          enhancement=3.0, figsize=(15, 10)):
    """
    Create and display RGB composites for all timesteps
    
    Parameters:
    - ds: xarray Dataset
    - var_name: name of the variable containing band data
    - rgb_bands: list of three band indices to use [r_idx, g_idx, b_idx]
    - enhancement: contrast enhancement factor
    - figsize: figure size
    """
    if var_name not in ds or 'time' not in ds.dims:
        print(f"Variable {var_name} not found or no time dimension in dataset")
        return
    
    n_times = ds.dims['time']
    
    # Create a grid of subplots
    n_cols = 3
    n_rows = (n_times + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    
    # Convert to 2D array of axes if it's 1D
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Flatten for easier iteration
    axes_flat = axes.flatten()
    
    # Process each timestep
    for t in range(n_times):
        try:
            # Extract the three bands for this timestep
            band_data = [
                ds[var_name].isel(time=t, band=b) for b in rgb_bands
            ]
            
            # Calculate min and max values for the RGB composite
            all_band_values = []
            for band in band_data:
                valid_values = band.values[~np.isnan(band.values)]
                if len(valid_values) > 0:
                    all_band_values.extend(valid_values)
            
            if all_band_values:
                data_min = np.min(all_band_values)
                data_max = np.max(all_band_values)
            else:
                data_min = data_max = 0
            
            # Apply contrast enhancement
            enhanced_bands = [enhance_contrast(band, enhancement) for band in band_data]
            
            # Create RGB composite
            rgb = np.zeros((*enhanced_bands[0].shape, 3))
            for i, band in enumerate(enhanced_bands):
                rgb[..., i] = band
            
            # Get timestamp if available
            timestamp = None
            if hasattr(ds, 't') and 't' in ds.coords:
                try:
                    timestamp = ds.t.isel(time=t, band=rgb_bands[0]).values
                except:
                    timestamp = f"Timestep {t}"
            else:
                timestamp = f"Timestep {t}"
            
            # Display the image
            axes_flat[t].imshow(rgb)
            
            # Create title with timestamp and min/max values
            title = f"{timestamp}\nMin: {data_min:.3f}, Max: {data_max:.3f}"
            axes_flat[t].set_title(title, fontsize=10)
            axes_flat[t].axis('off')
            
        except Exception as e:
            print(f"Error processing timestep {t}: {str(e)}")
            axes_flat[t].text(0.5, 0.5, f"Error: {str(e)[:50]}...", 
                            ha='center', va='center', transform=axes_flat[t].transAxes)
            axes_flat[t].set_title(f"Timestep {t}\nError", fontsize=10)
            axes_flat[t].axis('off')
    
    # Hide any unused subplots
    for i in range(n_times, len(axes_flat)):
        axes_flat[i].axis('off')
        axes_flat[i].set_visible(False)
    
    # Add a common title
    plt.suptitle(f"RGB Composites (R: Band {rgb_bands[0]}, G: Band {rgb_bands[1]}, B: Band {rgb_bands[2]})",
               fontsize=16)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.85)  # Make more space for the suptitle and individual titles
    plt.show()


In [5]:
# Function to process multiple zarr files
def process_multiple_zarr_files(directory_path, rgb_bands=[0, 1, 4], enhancement=3.0):
    """Process all zarr files in a directory"""
    
    # Find all zarr files
    zarr_files = find_zarr_files(directory_path)
    
    if not zarr_files:
        print(f"No .zarr files found in {directory_path}")
        return
    
    print(f"Found {len(zarr_files)} zarr files:")
    for i, file in enumerate(zarr_files):
        print(f"  {i+1}. {os.path.basename(file)}")
    
    # Process each file
    for zarr_file in zarr_files:
        print(f"\n{'='*60}")
        print(f"Processing: {os.path.basename(zarr_file)}")
        print(f"{'='*60}")
        
        # Open the zarr dataset
        ds = open_zarr_file(zarr_file)
        
        if ds is not None and 'time' in ds.dims:
            # Display information about timesteps
            print(f"\nNumber of timesteps: {ds.dims['time']}")
            
            # Show timestamps if available
            if hasattr(ds, 't') and 't' in ds.coords:
                print("Available timestamps:")
                for t in range(min(ds.dims['time'], 5)):  # Show up to 5 timestamps
                    try:
                        print(f"  [{t}] {ds.t.isel(time=t, band=0).values}")
                    except:
                        pass
            
            # Create RGB composites for all timesteps
            print(f"\nCreating RGB composites for {os.path.basename(zarr_file)}...")
            visualize_all_timesteps(
                ds, 
                var_name='Rad', 
                rgb_bands=rgb_bands,
                enhancement=enhancement,
                figsize=(16, 12)  # Increased height to accommodate min/max info
            )
        else:
            print("Dataset not found or has no time dimension")


In [ ]:
# Process all zarr files in directory
directory_path = '/Users/soehrle/Downloads/subset/'

print("PROCESSING ALL ZARR FILES IN DIRECTORY")
print("="*60)

process_multiple_zarr_files(
    directory_path=directory_path,
    rgb_bands=[0, 1, 4],
    enhancement=3.0
)